In [29]:
%pip install pandas
import pandas as pd

Note: you may need to restart the kernel to use updated packages.


In [30]:

# Loading our attrition data into a pandas dataframe
df = pd.read_csv(
    "../data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv"
)

In [31]:
# Attrition rate by each useful column
# AttritionRate is the share of employees in each group who left.

# Make a new column AttritionFlag that is True (1) for employees who left and False (0) for those who stayed.
df["AttritionFlag"] = df["Attrition"].eq("Yes")

# Compute the overall attrition rate to use as a baseline for comparison.
baseline_rate = df["AttritionFlag"].mean()
print(f"Overall attrition rate: {baseline_rate:.1%}")

# Identify columns that are not useful for attrition analysis.
constant_columns = [
    column for column in df.columns
    if df[column].nunique(dropna=False) <= 1
]
identifier_columns = ["EmployeeNumber"]
exclude_columns = {"Attrition", "AttritionFlag", *constant_columns, *identifier_columns}

# Group attrition rates by each useful column, and compare to the overall baseline rate.
results = []
for column in df.columns:
    # Skip columns that are not useful for attrition analysis.
    if column in exclude_columns:
        continue
    grouped_data = df[[column, "AttritionFlag"]].copy()
    group_column = column
    # If the column is numeric, create quartile bands to group the data.
    if pd.api.types.is_numeric_dtype(grouped_data[column]):
        group_column = f"{column}Band"
        grouped_data[group_column] = pd.qcut(
            grouped_data[column],
            q=4,
            duplicates="drop"
        )
    # Compute the attrition rate for each group and compare to the baseline.
    grouped_rates = (
        grouped_data
        .groupby(group_column, observed=False)["AttritionFlag"]
        .agg(
            Employees="size",
            Leavers="sum",
            AttritionRate="mean"
        )
        .reset_index()
    )

    # Add the column name to the results so we can identify which feature each group belongs to.
    grouped_rates.insert(0, "Feature", column)
    grouped_rates["RateVsBaseline"] = grouped_rates["AttritionRate"] - baseline_rate
    results.append(grouped_rates)

attrition_by_group = pd.concat(results, ignore_index=True)
# Format the attrition rates as percentages for easier reading.
attrition_by_group["AttritionRate"] = attrition_by_group["AttritionRate"].map(
    lambda rate: f"{rate:.1%}"
)
# Format the difference from baseline as a percentage with a sign to indicate whether the group is above or below the baseline.
attrition_by_group["RateVsBaseline"] = attrition_by_group["RateVsBaseline"].map(
    lambda rate: f"{rate:+.1%}"
)

# Sort the largest groups first so small, unstable groups do not dominate the view.
attrition_by_group = attrition_by_group.sort_values(
    ["Feature", "Employees"],
    ascending=[True, False]
)

# Pearson correlation with the binary AttritionFlag is a quick numeric screening tool.
numeric_columns = [
    column for column in df.select_dtypes(include="number").columns
    if column not in exclude_columns
]
# Compute the correlation of each numeric column with AttritionFlag, and sort by absolute value.
attrition_correlations = (
    df[numeric_columns]
    .corrwith(df["AttritionFlag"])
    .sort_values(key=lambda values: values.abs(), ascending=False)
    .rename("CorrelationWithAttrition")
    .to_frame()
)

print("\nGrouped attrition rates:")
display(attrition_by_group.head(30))
print("\nNumeric correlations with AttritionFlag:")
display(attrition_correlations)

Overall attrition rate: 16.1%



Grouped attrition rates:


,Feature,AgeBand,Employees,Leavers,AttritionRate,RateVsBaseline,BusinessTravel,DailyRateBand,Department,DistanceFromHomeBand,...,PerformanceRatingBand,RelationshipSatisfactionBand,StockOptionLevelBand,TotalWorkingYearsBand,TrainingTimesLastYearBand,WorkLifeBalanceBand,YearsAtCompanyBand,YearsInCurrentRoleBand,YearsSinceLastPromotionBand,YearsWithCurrManagerBand
1,Age,"(30.0, 36.0]",412,66,16.0%,-0.1%,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,Age,"(17.999, 30.0]",386,100,25.9%,+9.8%,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Age,"(43.0, 60.0]",347,42,12.1%,-4.0%,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Age,"(36.0, 43.0]",325,29,8.9%,-7.2%,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,BusinessTravel,NaN,1043,156,15.0%,-1.2%,Travel_Rarely,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,BusinessTravel,NaN,277,69,24.9%,+8.8%,Travel_Frequently,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,BusinessTravel,NaN,150,12,8.0%,-8.1%,Non-Travel,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,DailyRate,NaN,369,74,20.1%,+3.9%,NaN,"(101.999, 465.0]",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,DailyRate,NaN,367,59,16.1%,-0.0%,NaN,"(465.0, 802.0]",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,DailyRate,NaN,367,56,15.3%,-0.9%,NaN,"(802.0, 1157.0]",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Numeric correlations with AttritionFlag:


,CorrelationWithAttrition
TotalWorkingYears,-0.171063
JobLevel,-0.169105
YearsInCurrentRole,-0.160545
MonthlyIncome,-0.159840
Age,-0.159205
YearsWithCurrManager,-0.156199
StockOptionLevel,-0.137145
YearsAtCompany,-0.134392
JobInvolvement,-0.130016
JobSatisfaction,-0.103481


In [32]:
# Compute the correlation of each numeric column with AttritionFlag, and sort by absolute value.
correlation_data = df[numeric_columns].copy()
correlation_data["AttritionFlag"] = df["AttritionFlag"].astype(int)

correlation_data.corr()["AttritionFlag"].sort_values(key=lambda values: values.abs(), ascending=False)


AttritionFlag               1.000000
TotalWorkingYears          -0.171063
JobLevel                   -0.169105
YearsInCurrentRole         -0.160545
MonthlyIncome              -0.159840
Age                        -0.159205
YearsWithCurrManager       -0.156199
StockOptionLevel           -0.137145
YearsAtCompany             -0.134392
JobInvolvement             -0.130016
JobSatisfaction            -0.103481
EnvironmentSatisfaction    -0.103369
DistanceFromHome            0.077924
WorkLifeBalance            -0.063939
TrainingTimesLastYear      -0.059478
DailyRate                  -0.056652
RelationshipSatisfaction   -0.045872
NumCompaniesWorked          0.043494
YearsSinceLastPromotion    -0.033019
Education                  -0.031373
MonthlyRate                 0.015170
PercentSalaryHike          -0.013478
HourlyRate                 -0.006846
PerformanceRating           0.002889
Name: AttritionFlag, dtype: float64

In [ ]:
# Create one grouped correlation DataFrame for every analyzed feature.
attrition_correlations_by_group = {}

for feature in attrition_correlations.index:
    feature_data = df[[feature, "AttritionFlag"]].copy()

    if pd.api.types.is_numeric_dtype(feature_data[feature]):
        feature_data["Group"] = pd.qcut(
            feature_data[feature],
            q=4,
            duplicates="drop"
        )
    else:
        feature_data["Group"] = feature_data[feature].fillna("Missing")

    group_rows = []
    for group_value, group in feature_data.groupby(
        "Group",
        observed=False,
        sort=False
    ):
        group_membership = feature_data["Group"].eq(group_value).astype(int)
        group_rows.append({
            "Feature": feature,
            "Group": group_value,
            "Employees": len(group),
            "Leavers": int(group["AttritionFlag"].sum()),
            "AttritionRate": group["AttritionFlag"].mean(),
            "RateVsBaseline": group["AttritionFlag"].mean() - baseline_rate,
            "GroupAttritionCorrelation": group_membership.corr(
                feature_data["AttritionFlag"].astype(int)
            ),
        })

    attrition_correlations_by_group[feature] = pd.DataFrame(group_rows)

In [ ]:
for feature, correlation_df in attrition_correlations_by_group.items():
    print(f"\n{feature}")
    sorted_correlation_df = correlation_df.sort_values(
        "Group",
        ascending=True
    ).reset_index(drop=True).copy()
    sorted_correlation_df["AttritionRate"] = sorted_correlation_df[
        "AttritionRate"
    ].map(lambda rate: f"{rate:.1%}")
    sorted_correlation_df["RateVsBaseline"] = sorted_correlation_df[
        "RateVsBaseline"
    ].map(lambda rate: f"{rate:+.1%}")
    display(sorted_correlation_df)


TotalWorkingYears


,Feature,Group,Employees,Leavers,AttritionRate,RateVsBaseline,GroupAttritionCorrelation
0,TotalWorkingYears,"(-0.001, 6.0]",441,113,0.256236,0.095011,0.169141
1,TotalWorkingYears,"(6.0, 10.0]",482,69,0.143154,-0.018071,-0.034323
2,TotalWorkingYears,"(10.0, 15.0]",191,24,0.125654,-0.035570,-0.037379
3,TotalWorkingYears,"(15.0, 40.0]",356,31,0.087079,-0.074146,-0.113981



JobLevel


,Feature,Group,Employees,Leavers,AttritionRate,RateVsBaseline,GroupAttritionCorrelation
0,JobLevel,"(0.999, 2.0]",1077,195,0.181058,0.019834,0.089286
1,JobLevel,"(2.0, 3.0]",218,32,0.146789,-0.014435,-0.016380
2,JobLevel,"(3.0, 5.0]",175,10,0.057143,-0.104082,-0.104045



YearsInCurrentRole


...

,Feature,Group,Employees,Leavers,AttritionRate,RateVsBaseline,GroupAttritionCorrelation
0,PercentSalaryHike,"(10.999, 12.0]",408,74,0.181373,0.020148,0.033960
1,PercentSalaryHike,"(12.0, 14.0]",410,58,0.141463,-0.019761,-0.033420
2,PercentSalaryHike,"(14.0, 18.0]",350,59,0.168571,0.007347,0.011168
3,PercentSalaryHike,"(18.0, 25.0]",302,46,0.152318,-0.008907,-0.012316



HourlyRate


,Feature,Group,Employees,Leavers,AttritionRate,RateVsBaseline,GroupAttritionCorrelation
0,HourlyRate,"(29.999, 48.0]",375,57,0.152000,-0.009224,-0.014680
1,HourlyRate,"(48.0, 66.0]",378,68,0.179894,0.018670,0.029870
2,HourlyRate,"(66.0, 83.75]",349,51,0.146132,-0.015093,-0.022900
3,HourlyRate,"(83.75, 100.0]",368,61,0.165761,0.004536,0.007129



PerformanceRating


,Feature,Group,Employees,Leavers,AttritionRate,RateVsBaseline,GroupAttritionCorrelation
0,PerformanceRating,"(2.999, 4.0]",1470,237,0.161224,0.0,NaN
